In [1]:
# =============================================================
#   FULL ENHANCEMENT PIPELINE
#   Video → Frames → Grayscale → Median → CLAHE → Laplacian
# =============================================================


import cv2
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


video_path = "poth1.mov"
frame_interval = 10 


folders = {
    "raw": "frames/raw",
    "grayscale": "frames/grayscale",
    "median": "frames/enhanced/median",
    "clahe": "frames/enhanced/clahe",
    "laplacian": "frames/enhanced/laplacian",
    "comparisons": "frames/enhanced/comparisons"
}

for key, folder in folders.items():
    if not os.path.exists(folder):
        os.makedirs(folder)
        print(f"{key.capitalize()} folder created!")
    else:
        print(f"{key.capitalize()} folder already exists!")


def calculate_entropy(image):
    histogram, _ = np.histogram(image.flatten(), bins=256, range=(0,256))
    histogram = histogram / histogram.sum()
    histogram = histogram[histogram > 0]
    entropy = -np.sum(histogram * np.log2(histogram))
    return entropy

def calculate_psnr(original, enhanced):
    mse = np.mean((original - enhanced) ** 2)
    if mse == 0:
        return 100
    return 20 * np.log10(255.0 / np.sqrt(mse))


if not os.path.exists(video_path):
    raise FileNotFoundError(f"Video file not found: {video_path}")

cap = cv2.VideoCapture(video_path)
frame_count = 0
saved_count = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break
    if frame_count % frame_interval == 0:
        file_path = os.path.join(folders["raw"], f"frame_{saved_count:04d}.jpg")
        cv2.imwrite(file_path, frame)
        saved_count += 1
    frame_count += 1

cap.release()
print(f"Frame extraction done. Total frames saved: {saved_count}")


frame_files = sorted(os.listdir(folders["raw"]))
for fname in frame_files:
    img_path = os.path.join(folders["raw"], fname)
    img = cv2.imread(img_path)
    if img is None:
        continue
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    save_path = os.path.join(folders["grayscale"], fname)
    cv2.imwrite(save_path, gray)
    # Save histogram as PNG (optional)
    hist_path = os.path.join(folders["grayscale"], f"{os.path.splitext(fname)[0]}_hist.png")
    plt.figure()
    plt.hist(gray.ravel(), bins=256, range=(0, 256))
    plt.savefig(hist_path)
    plt.close()
print("Grayscale conversion completed.")


median_results = []
for fname in frame_files:
    gray_path = os.path.join(folders["grayscale"], fname)
    gray = cv2.imread(gray_path, cv2.IMREAD_GRAYSCALE)
    if gray is None:
        continue
    median = cv2.medianBlur(gray, 11)
    save_path = os.path.join(folders["median"], fname)
    cv2.imwrite(save_path, median)

    mean_before = np.mean(gray)
    mean_after = np.mean(median)
    std_before = np.std(gray)
    std_after = np.std(median)
    entropy_before = calculate_entropy(gray)
    entropy_after = calculate_entropy(median)
    psnr_value = calculate_psnr(gray, median)

    median_results.append([fname, mean_before, mean_after, std_before, std_after, entropy_before, entropy_after, psnr_value])

# Save median metrics
df_median = pd.DataFrame(median_results, columns=["Frame", "Mean Before", "Mean After", "Std Before", "Std After", "Entropy Before", "Entropy After", "PSNR"])
df_median.to_csv(os.path.join(folders["median"], "median_metrics.csv"), index=False)
print("Median filtering completed.")


clahe_results = []
clahe_obj = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))

for fname in frame_files:
    gray_path = os.path.join(folders["grayscale"], fname)
    img = cv2.imread(gray_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        continue
    clahe_img = clahe_obj.apply(img)
    save_path = os.path.join(folders["clahe"], fname)
    cv2.imwrite(save_path, clahe_img)

    mean_before = np.mean(img)
    mean_after = np.mean(clahe_img)
    std_before = np.std(img)
    std_after = np.std(clahe_img)
    entropy_before = calculate_entropy(img)
    entropy_after = calculate_entropy(clahe_img)
    psnr_value = calculate_psnr(img, clahe_img)

    clahe_results.append([fname, mean_before, mean_after, std_before, std_after, entropy_before, entropy_after, psnr_value])

df_clahe = pd.DataFrame(clahe_results, columns=["Frame", "Mean Before", "Mean After", "Std Before", "Std After", "Entropy Before", "Entropy After", "PSNR"])
df_clahe.to_csv(os.path.join(folders["clahe"], "clahe_metrics.csv"), index=False)
print("CLAHE enhancement completed.")


for fname in frame_files:
    gray_path = os.path.join(folders["clahe"], fname)
    img = cv2.imread(gray_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        continue
    lap = cv2.Laplacian(img, cv2.CV_64F)
    lap = cv2.convertScaleAbs(lap)
    save_path = os.path.join(folders["laplacian"], fname)
    cv2.imwrite(save_path, lap)
print("Laplacian enhancement completed.")


for fname in frame_files:
    gray       = cv2.imread(os.path.join(folders["grayscale"], fname), cv2.IMREAD_GRAYSCALE)
    median_img = cv2.imread(os.path.join(folders["median"], fname), cv2.IMREAD_GRAYSCALE)
    clahe_img  = cv2.imread(os.path.join(folders["clahe"], fname), cv2.IMREAD_GRAYSCALE)
    lap_img    = cv2.imread(os.path.join(folders["laplacian"], fname), cv2.IMREAD_GRAYSCALE)

    stages = [
        ("Grayscale", gray),
        ("Median Filtered", median_img),
        ("CLAHE Enhanced", clahe_img),
        ("Laplacian Enhanced", lap_img),
    ]
    stages = [(label, img) for label, img in stages if img is not None]
    n = len(stages)
    if n == 0:
        continue

    fig, axes = plt.subplots(1, n, figsize=(5*n,5))
    if n == 1:
        axes = [axes]
    for ax, (label, img) in zip(axes, stages):
        ax.imshow(img, cmap="gray")
        ax.set_title(label)
        ax.axis("off")
    plt.tight_layout()
    save_path = os.path.join(folders["comparisons"], fname.replace(".jpg","_comparison.jpg"))
    plt.savefig(save_path, bbox_inches="tight", dpi=150)
    plt.close()

print("Pipeline completed. All outputs saved in relevant folders.")

Raw folder created!
Grayscale folder created!
Median folder created!
Clahe folder created!
Laplacian folder created!
Comparisons folder created!
Frame extraction done. Total frames saved: 33
Grayscale conversion completed.
Median filtering completed.
CLAHE enhancement completed.
Laplacian enhancement completed.
Pipeline completed. All outputs saved in relevant folders.
